[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_k * num_kv_heads)
        self.W_v = nn.Linear(d_model, d_k * num_kv_heads)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_k

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k)
        K = K.view(batch_size, seq_len, self.num_kv_heads, self.d_k)
        V = V.view(batch_size, seq_len, self.num_kv_heads, self.d_k)
        group_size = self.num_heads // self.num_kv_heads

        Q = Q.transpose(1,2) # shape: (batch_size, self.num_heads, seq_len, self.d_k)
        K = K.transpose(1,2)
        V = V.transpose(1,2)
        Q = Q.contiguous().view(batch_size, self.num_kv_heads, group_size, seq_len, self.d_k)
        K = K.unsqueeze(2) # shape: (batch_size, self.num_kv_heads, 1, seq_len, self.d_k)
        V = V.unsqueeze(2) # shape: (batch_size, self.num_kv_heads, 1, seq_len, self.d_k)

        # broadcast within group
        context_vector = torch.softmax((Q @ K.transpose(-1,-2)) / math.sqrt(self.d_k), dim = -1) @ V # shape (batch_size, self.num_kv_heads, group_size, seq_len, self.d_k)
        context_vector = context_vector.contiguous().view(batch_size, self.num_heads, seq_len, self.d_k)
        context_vector = context_vector.transpose(1,2)
        context_vector = context_vector.contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_o(context_vector)

In [9]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [10]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (2.4ms)
  ✅ [2/5] nn.Linear with correct shapes (0.6ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.4ms)
  ✅ [4/5] KV heads are shared correctly (5.1ms)
  ✅ [5/5] Gradient flow (13.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (22.5ms total)
  Progress saved. Run status() to see your dashboard.

